# Module 5 — Xuất báo cáo PDF chuyên nghiệp (reportlab)

Notebook đọc dữ liệu sạch `Data/output/du_lieu_sach.csv` và tạo báo cáo PDF `Report/BaoCao_SCADA.pdf` gồm:

- **Trang bìa**: tiêu đề, tên bộ dữ liệu, tổng số bản ghi
- **Chương 1**: bảng các chỉ số chính (có màu nền tiêu đề và viền)
- **Chương 2**: đường cong công suất (chèn ảnh) + giải thích
- **Chương 3**: biểu đồ sản lượng theo tháng (chèn ảnh)
- **Chương 4**: kết luận và kiến nghị

**Lưu ý:** để tránh lỗi font tiếng Việt trong PDF, **nội dung PDF dùng chữ không dấu**. Biểu đồ vẽ bằng `matplotlib` với backend `Agg`, lưu PNG tạm rồi chèn vào PDF, sau đó **xóa file PNG tạm**.

## Bước 1 — Đọc dữ liệu, tính chỉ số, chuẩn bị thư viện

In [1]:
import os
import matplotlib
matplotlib.use("Agg")  # backend không cần màn hình, phù hợp khi chỉ lưu file ảnh
import matplotlib.pyplot as plt
import pandas as pd

from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak,
)

CSV_INPUT = "../Data/output/du_lieu_sach.csv"
PDF_OUTPUT = "../Report/BaoCao_SCADA.pdf"
TEN_BO_DU_LIEU = "T1.csv - Du lieu SCADA tua bin gio"
PHUT_MOI_DIEM = 10  # mỗi bản ghi cách nhau 10 phút

# Đọc dữ liệu sạch
df = pd.read_csv(CSV_INPUT, index_col="thoi_gian", parse_dates=True)

# Cờ điểm nghi sự cố: gió > 5 m/s và công suất < 50% công suất lý thuyết
df["nghi_su_co"] = (df["toc_do_gio"] > 5) & (df["cong_suat_kw"] < 0.5 * df["cong_suat_ly_thuyet"])

# Các chỉ số chính
tong_ban_ghi = len(df)
cong_suat_tb = round(df["cong_suat_kw"].mean(), 2)
toc_do_gio_tb = round(df["toc_do_gio"].mean(), 2)
so_diem_su_co = int(df["nghi_su_co"].sum())
tong_gio_su_co = round(so_diem_su_co * PHUT_MOI_DIEM / 60, 2)

print(f"Tong ban ghi     : {tong_ban_ghi}")
print(f"Cong suat TB (kW): {cong_suat_tb}")
print(f"Toc do gio TB    : {toc_do_gio_tb}")
print(f"So diem su co    : {so_diem_su_co}")
print(f"Tong gio su co   : {tong_gio_su_co}")

Tong ban ghi     : 48835
Cong suat TB (kW): 1225.7
Toc do gio TB    : 7.22
So diem su co    : 2219
Tong gio su co   : 369.83


## Bước 2 — Vẽ 2 biểu đồ và lưu ra file PNG tạm

Biểu đồ vẽ bằng matplotlib (backend Agg), lưu tạm để lát chèn vào PDF rồi xóa.

In [2]:
# Đường dẫn file PNG tạm (sẽ xóa sau khi chèn vào PDF)
PNG_DUONG_CONG = "_tmp_duong_cong_cong_suat.png"
PNG_SAN_LUONG = "_tmp_san_luong_thang.png"

# --- Biểu đồ 1: đường cong công suất, tô đỏ điểm nghi sự cố ---
binh_thuong = ~df["nghi_su_co"]
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df.loc[binh_thuong, "toc_do_gio"], df.loc[binh_thuong, "cong_suat_kw"],
           s=4, alpha=0.3, color="#4C9BE8", label="Binh thuong")
ax.scatter(df.loc[df["nghi_su_co"], "toc_do_gio"], df.loc[df["nghi_su_co"], "cong_suat_kw"],
           s=4, alpha=0.5, color="red", label=f"Nghi su co ({so_diem_su_co} diem)")
ax.set_title("Duong cong cong suat tua bin gio")
ax.set_xlabel("Toc do gio (m/s)")
ax.set_ylabel("Cong suat thuc (kW)")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(PNG_DUONG_CONG, dpi=120)
plt.close(fig)

# --- Biểu đồ 2: sản lượng theo tháng + tốc độ gió trung bình ---
theo_thang = df.resample("ME").agg(
    tong_cong_suat=("cong_suat_kw", "sum"),
    toc_do_gio_tb=("toc_do_gio", "mean"),
)
nhan_thang = theo_thang.index.strftime("%m/%Y")
san_luong_mwh = theo_thang["tong_cong_suat"] * PHUT_MOI_DIEM / 60 / 1000

fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.bar(nhan_thang, san_luong_mwh, color="#4C9BE8", label="San luong (MWh)")
ax1.set_xlabel("Thang")
ax1.set_ylabel("San luong (MWh)", color="#2A6BB0")
ax1.tick_params(axis="x", rotation=45)
ax2 = ax1.twinx()
ax2.plot(nhan_thang, theo_thang["toc_do_gio_tb"], color="orange", marker="o", linewidth=2,
         label="Toc do gio TB (m/s)")
ax2.set_ylabel("Toc do gio TB (m/s)", color="darkorange")
ax1.set_title("San luong theo thang va toc do gio trung binh")
fig.tight_layout()
fig.savefig(PNG_SAN_LUONG, dpi=120)
plt.close(fig)

print("Da luu 2 file PNG tam:", PNG_DUONG_CONG, "va", PNG_SAN_LUONG)

Da luu 2 file PNG tam: _tmp_duong_cong_cong_suat.png va _tmp_san_luong_thang.png


## Bước 3 — Dựng nội dung PDF (trang bìa + 4 chương) và xuất file

In [3]:
# Tạo folder Report nếu chưa có
os.makedirs(os.path.dirname(PDF_OUTPUT), exist_ok=True)

# Khung tài liệu A4 với lề hợp lý
doc = SimpleDocTemplate(
    PDF_OUTPUT, pagesize=A4,
    topMargin=2 * cm, bottomMargin=2 * cm, leftMargin=2 * cm, rightMargin=2 * cm,
)

# Định nghĩa style chữ (dùng font mặc định Helvetica -> nội dung phải không dấu)
styles = getSampleStyleSheet()
style_tieu_de = ParagraphStyle("TieuDe", parent=styles["Title"], fontSize=24, leading=30)
style_bia_phu = ParagraphStyle("BiaPhu", parent=styles["Normal"], fontSize=13,
                               alignment=TA_CENTER, leading=20)
style_chuong = ParagraphStyle("Chuong", parent=styles["Heading1"], fontSize=16,
                              spaceBefore=12, spaceAfter=10, textColor=colors.HexColor("#1F4E79"))
style_than = ParagraphStyle("Than", parent=styles["Normal"], fontSize=11, leading=16)

# Danh sách các phần tử (flowables) của tài liệu
noi_dung = []

# ===== TRANG BÌA =====
noi_dung.append(Spacer(1, 5 * cm))
noi_dung.append(Paragraph("BAO CAO PHAN TICH DU LIEU DIEN GIO", style_tieu_de))
noi_dung.append(Spacer(1, 1 * cm))
noi_dung.append(Paragraph(f"Bo du lieu: {TEN_BO_DU_LIEU}", style_bia_phu))
noi_dung.append(Paragraph(f"Tong so ban ghi: {tong_ban_ghi:,}", style_bia_phu))
noi_dung.append(Paragraph("Bao cao tu dong tao bang Python & reportlab", style_bia_phu))
noi_dung.append(PageBreak())

# ===== CHUONG 1: BANG CHI SO CHINH =====
noi_dung.append(Paragraph("Chuong 1. Cac chi so chinh", style_chuong))
noi_dung.append(Paragraph(
    "Bang duoi tong hop cac chi so van hanh chinh cua tua bin trong ky bao cao.", style_than))
noi_dung.append(Spacer(1, 0.4 * cm))

du_lieu_bang = [
    ["Chi so", "Gia tri"],
    ["Tong so ban ghi", f"{tong_ban_ghi:,}"],
    ["Cong suat trung binh (kW)", f"{cong_suat_tb:,.2f}"],
    ["Toc do gio trung binh (m/s)", f"{toc_do_gio_tb:,.2f}"],
    ["So diem nghi su co", f"{so_diem_su_co:,}"],
    ["Tong thoi gian su co (gio)", f"{tong_gio_su_co:,.2f}"],
]
bang = Table(du_lieu_bang, colWidths=[9 * cm, 6 * cm])
bang.setStyle(TableStyle([
    # Hàng tiêu đề: nền xanh đậm, chữ trắng, in đậm
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1F4E79")),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
    ("FONTSIZE", (0, 0), (-1, -1), 11),
    ("ALIGN", (1, 0), (1, -1), "RIGHT"),
    ("ALIGN", (1, 0), (1, 0), "CENTER"),
    # Viền cho toàn bảng và các dòng xen kẽ màu nền nhạt
    ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
    ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#EAF1F8")]),
    ("TOPPADDING", (0, 0), (-1, -1), 6),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
]))
noi_dung.append(bang)
noi_dung.append(PageBreak())

# ===== CHUONG 2: DUONG CONG CONG SUAT =====
noi_dung.append(Paragraph("Chuong 2. Duong cong cong suat", style_chuong))
noi_dung.append(Paragraph(
    "Bieu do phan tan the hien quan he giua toc do gio (truc X) va cong suat thuc (truc Y). "
    "Cac diem mau do la diem nghi su co: gio manh (> 5 m/s) nhung cong suat thap bat thuong "
    "(< 50% cong suat ly thuyet), co the do tua bin dung may, bao tri hoac loi cam bien.",
    style_than))
noi_dung.append(Spacer(1, 0.4 * cm))
noi_dung.append(Image(PNG_DUONG_CONG, width=15 * cm, height=9.375 * cm))
noi_dung.append(PageBreak())

# ===== CHUONG 3: SAN LUONG THEO THANG =====
noi_dung.append(Paragraph("Chuong 3. San luong theo thang", style_chuong))
noi_dung.append(Paragraph(
    "Cot the hien tong san luong dien moi thang (MWh); duong mau cam la toc do gio trung binh "
    "thang (truc phai). Thong thuong hai dai luong di cung chieu: thang gio manh cho san luong cao.",
    style_than))
noi_dung.append(Spacer(1, 0.4 * cm))
noi_dung.append(Image(PNG_SAN_LUONG, width=15 * cm, height=9.375 * cm))
noi_dung.append(PageBreak())

# ===== CHUONG 4: KET LUAN VA KIEN NGHI =====
noi_dung.append(Paragraph("Chuong 4. Ket luan va kien nghi", style_chuong))
ty_le_su_co = so_diem_su_co / tong_ban_ghi * 100
noi_dung.append(Paragraph(
    f"Trong ky bao cao, he thong ghi nhan {tong_ban_ghi:,} ban ghi, trong do co "
    f"{so_diem_su_co:,} diem nghi su co (chiem {ty_le_su_co:.2f}%), tuong ung khoang "
    f"{tong_gio_su_co:,.2f} gio tua bin hoat dong kem hieu qua du dieu kien gio thuan loi.",
    style_than))
noi_dung.append(Spacer(1, 0.3 * cm))
noi_dung.append(Paragraph("Kien nghi:", style_than))
noi_dung.append(Paragraph(
    "1. Ra soat lich bao tri va nhat ky dung may tai cac thoi diem nghi su co de xac dinh nguyen nhan.",
    style_than))
noi_dung.append(Paragraph(
    "2. Kiem tra cam bien cong suat va cam bien gio, doi chieu voi cong suat ly thuyet.",
    style_than))
noi_dung.append(Paragraph(
    "3. Theo doi cac thang co san luong thap bat thuong du toc do gio cao de phat hien su co keo dai.",
    style_than))

# Xuất PDF
doc.build(noi_dung)
print(f"Da tao bao cao PDF: {PDF_OUTPUT}")

Da tao bao cao PDF: ../Report/BaoCao_SCADA.pdf


## Bước 4 — Xóa file PNG tạm và kiểm chứng

In [4]:
# Sau khi ảnh đã được chèn vào PDF, xóa các file PNG tạm
for f in (PNG_DUONG_CONG, PNG_SAN_LUONG):
    if os.path.exists(f):
        os.remove(f)
        print(f"Da xoa file tam: {f}")

# Kiểm chứng file PDF đã tạo
kich_thuoc_kb = os.path.getsize(PDF_OUTPUT) / 1024
print(f"\nFile PDF: {PDF_OUTPUT}")
print(f"Kich thuoc: {kich_thuoc_kb:.1f} KB")
print("File PNG tam con lai:", [f for f in (PNG_DUONG_CONG, PNG_SAN_LUONG) if os.path.exists(f)])

Da xoa file tam: _tmp_duong_cong_cong_suat.png
Da xoa file tam: _tmp_san_luong_thang.png

File PDF: ../Report/BaoCao_SCADA.pdf
Kich thuoc: 181.7 KB
File PNG tam con lai: []
